In [ ]:
import numpy as np
import pandas as pd
import json
import datetime
import matplotlib.pyplot as plt

In [ ]:
with open('data.json','r') as file:
    data = json.load(file)

In [ ]:
len(data)

In [ ]:
visited = set()
encoded = []
for i, record in enumerate(data):
    key = f"{record['WORKER_ID']}/{record['WORK']['taskId']}"
    if key in visited or len(key)!=41: continue
    visited.add(key)
    work=record['WORK']
    for j, row in enumerate(work['items']):
        encoded.append({
            "worker_id": work['workerId'],
            "start_time": datetime.datetime.fromtimestamp(round(row['startTime']/1000)),
            "end_time": datetime.datetime.fromtimestamp(round(row['endTime']/1000)),
            "group": work['group'],
            "paramoji1": [int(x) for x in row['face1']['paramoji']],
            "paramoji2": [int(x) for x in row['face2']['paramoji']],
            "value1": row['value1'],
            "value2": row['value2'],
        })
encoded= pd.DataFrame(encoded)

In [ ]:
encoded

In [ ]:
def extract_group(g):
    input = np.concatenate((
        np.array(encoded.query("group==@g")['paramoji1'].to_list()),
        np.array(encoded.query("group==@g")['paramoji2'].to_list())
    ))-50
    output = np.concatenate((
        np.array(encoded.query("group==@g")['value1'].to_list()),
        np.array(encoded.query("group==@g")['value2'].to_list())
    ))*50 + 50
    return input, output

In [ ]:
coeffs = []
groups = sorted(set(encoded['group']))
for g in groups:
    input, output = extract_group(g)
    input= np.vstack((input.T, np.ones(output.shape))).T
    mc = np.linalg.lstsq(input, output, rcond=None)[0]
    coeffs.append(mc[:5])
coeffs = np.array(coeffs)
coeffs = pd.DataFrame(coeffs, index=groups)
coeffs

In [ ]:
def proj(u,v):
    return u.T @ v / (u.T @ u) * u

In [ ]:
ortho = coeffs.copy()
for i in range(ortho.shape[0]):
    for j in range(i+1,ortho.shape[0]):
        ortho.iloc[j,:] = ortho.iloc[j,:] - proj(ortho.iloc[i,:], ortho.iloc[j,:])
ortho

In [ ]:
ortho.plot()

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format
print(coeffs.to_latex())

In [ ]:
print(ortho.to_latex())